In [16]:
import pandas as pd

In [17]:
orderlines_df = pd.read_csv("data/raw/orderlines.csv")
products_df = pd.read_csv("data/raw/products.csv")
orders_df = pd.read_csv("data/raw/orders.csv")
brands_df = pd.read_csv("data/raw/brands.csv")

In [18]:
# product_id is not used -> drop
orderlines_df = orderlines_df.drop(columns = "product_id")

# promo_price is not helpful because many prices are wrong and we dont know the time of the promo -> drop
products_df = products_df.drop(columns = "promo_price")

# drop duplicate products
products_df = products_df.drop_duplicates()

# drop all products with Null price
# products_df = products_df.dropna(subset = "price")

# rename the columns of orderlines_df
orderlines_df.columns = ["id", "order_id", "product_quantity", "sku", "unit_price", "orderlines_date"]

# change orderlines_date to datetime
orderlines_df["orderlines_date"] = pd.to_datetime(orderlines_df["orderlines_date"])

# all products with Null description get the name as description
products_df.loc[products_df["desc"].isna(), "desc"] = \
    products_df.loc[products_df["desc"].isna(), "name"]

# replace all , to . in type and convert it to numeric
products_df["type"] = products_df["type"].str.replace(",", ".")
products_df["type"] = pd.to_numeric(products_df["type"])

In [19]:
# creating a new column price_clean to include the corrected price
orderlines_df["unit_price"] = (
    orderlines_df["unit_price"]
    .str.replace(r"\.(?=.*\.)", "", regex=True)
    .astype(float)
)
#  Converting to float & rounding decimal to 2
orderlines_df["unit_price"] = pd.to_numeric(orderlines_df["unit_price"], errors='coerce').round(2)

In [20]:
# count unit_price per sku
counts = orderlines_df.groupby("sku")["unit_price"].transform("count")

# creating median_price column only if count > 2
orderlines_df["median_price"] = orderlines_df.groupby("sku")["unit_price"].transform("median")
orderlines_df.loc[counts <= 2, "median_price"] = None

# creating price_ratio column
orderlines_df["price_ratio"] = orderlines_df["unit_price"] / orderlines_df["median_price"]

# every unit_price under /2 or above *2 is filtered out
mask = (orderlines_df["price_ratio"] < 2) & (orderlines_df["price_ratio"] > 0.5)

# only orders with the filtered price_ratio will be kept
valid_orders = mask.groupby(orderlines_df["order_id"]).all()

# creating filtered_df
filtered_df = orderlines_df[orderlines_df["order_id"].isin(valid_orders[valid_orders].index)]

In [21]:
# merge products_df and the filtered_df
products_merged = products_df.merge(filtered_df, on = "sku", how = "right")
products_merged = products_merged.dropna(subset=["order_id"])

# drop all orders with at least 1 Null price
valid_orders = products_merged["price"].notna().groupby(products_merged["order_id"]).all()
products_merged = products_merged[
    products_merged["order_id"].isin(valid_orders[valid_orders].index)
]


In [22]:
# all prices that ends with .[digit][digit] and without . are right
# creating column for wrong prices
products_merged["false_price"] = ~(
        (~ products_merged["price"].str.contains(r"\.")) |
        (products_merged["price"].str.contains(r"\.\d\d$"))
)

In [23]:
# creating a new column price_clean to include the corrected price
products_merged["price"] = (
    products_merged["price"]
    .str.replace(r"\.(?=.*\.)", "", regex=True)
    .astype(float)
)
#  Converting to float & rounding decimal to 2
products_merged["price"] = pd.to_numeric(products_merged["price"], errors='coerce').round(2)


In [24]:
# count unit_price per sku
counts = orderlines_df.groupby("sku")["unit_price"].transform("count")

# creating median_price column only if count is not 2
# problem are 2 unit_price with huge difference like 10 and 10000 -> what is the median?
orderlines_df["median_price"] = orderlines_df.groupby("sku")["unit_price"].transform("median")
orderlines_df.loc[counts == 2, "median_price"] = None

# creating price_ratio column from unit_price / median_price
orderlines_df["price_ratio"] = orderlines_df["unit_price"] / orderlines_df["median_price"]

# every unit_price under /2 or above *2 is filtered out
mask = (orderlines_df["price_ratio"] < 2) & (orderlines_df["price_ratio"] > 0.5)

# only orders with the filtered price_ratio will be kept
valid_orders = mask.groupby(orderlines_df["order_id"]).all()

# creating filtered_df that only keep valid orders, depending on the price_ratio filter
filtered_df = orderlines_df[orderlines_df["order_id"].isin(valid_orders[valid_orders].index)]

In [25]:
# creating ratio column: ratio of price / median_price
products_merged["ratio"] = products_merged["price"] / products_merged["median_price"]

In [26]:
products_merged.loc[
    (~products_merged["false_price"])
][["sku", "name", "price", "median_price", "unit_price", "ratio", "false_price"]].groupby("sku").max()

,name,price,median_price,unit_price,ratio,false_price
sku,,,,,,
8MO0003-A,Open - 8Mobility iSlice Micro SD adapter for M...,35.00,12.85,12.85,2.723735,False
8MO0007,8Mobility iSlice Micro SD adapter for Macbook ...,35.00,19.99,23.99,1.750875,False
8MO0008,8Mobility iSlice Micro SD Adapter Macbook Pro ...,35.00,19.99,23.99,1.750875,False
8MO0009,8Mobility iSlice Micro SD Adapter for Macbook ...,35.00,16.99,23.99,2.060035,False
8MO0010,8Mobility iSlice Micro SD Adapter for MacBook ...,35.00,19.99,23.99,1.750875,False
...,...,...,...,...,...,...
ZAG0037,"Zagg iPad Case Rugged Messenger 97 ""(2017) Bla...",29.99,9.99,9.99,3.002002,False
ZAG0038,Zagg Glass + X Clear Screen Protector iPhone,29.99,24.99,24.99,1.200080,False
ZAG0040,Zagg Glass Screen Protector + Contour Contour ...,39.99,29.99,29.99,1.333444,False


In [27]:
products_merged.loc[
    (products_merged["false_price"])
][["sku", "name", "price", "median_price", "unit_price", "ratio", "false_price"]].groupby("sku").max()

,name,price,median_price,unit_price,ratio,false_price
sku,,,,,,
ADN0049,Adonit Jot Mini 3.0 Pointer Black,25.20,19.99,19.99,1.260630,True
ADN0050,Adonit Jot Mini 3.0 Pointer Silver,25.20,19.99,19.99,1.260630,True
ADN0051,Adonit Jot Mini 3.0 Pointer Gold,25.20,19.99,19.99,1.260630,True
ADN0052,Adonit Jot Mini Rose Gold Pointer 3.0,25.20,19.99,19.99,1.260630,True
ADN0053,Adonit Jot Dash 3 Pointer Silver,60.50,44.09,49.99,1.372193,True
...,...,...,...,...,...,...
WDT0400,WD My Cloud Home 3TB USB 3.0,2049.95,204.99,212.79,10.000244,True
WIK0007,Wikango 600 Spanish warning radars,89.90,79.99,79.99,1.123890,True
WIK0008,Wikango 700 Spanish warning radars battery,109.90,99.99,99.99,1.099110,True


In [28]:
# the price of all products with false_price that are over 9 times of the median_price are divided by 10 until the ratio is under 9
while ((products_merged["ratio"] >= 9) & (products_merged["false_price"])).any():
    mask = (products_merged["ratio"] >= 9) & (products_merged["false_price"])

    products_merged.loc[mask, "price"] = products_merged.loc[mask, "price"] / 10

    products_merged["ratio"] = products_merged["price"] / products_merged["median_price"]

In [29]:
products_merged[["sku", "name", "price", "median_price", "unit_price", "ratio", "false_price"]].groupby("sku").max()

,name,price,median_price,unit_price,ratio,false_price
sku,,,,,,
8MO0003-A,Open - 8Mobility iSlice Micro SD adapter for M...,35.00,12.85,12.85,2.723735,False
8MO0007,8Mobility iSlice Micro SD adapter for Macbook ...,35.00,19.99,23.99,1.750875,False
8MO0008,8Mobility iSlice Micro SD Adapter Macbook Pro ...,35.00,19.99,23.99,1.750875,False
8MO0009,8Mobility iSlice Micro SD Adapter for Macbook ...,35.00,16.99,23.99,2.060035,False
8MO0010,8Mobility iSlice Micro SD Adapter for MacBook ...,35.00,19.99,23.99,1.750875,False
...,...,...,...,...,...,...
ZAG0037,"Zagg iPad Case Rugged Messenger 97 ""(2017) Bla...",29.99,9.99,9.99,3.002002,False
ZAG0038,Zagg Glass + X Clear Screen Protector iPhone,29.99,24.99,24.99,1.200080,False
ZAG0040,Zagg Glass Screen Protector + Contour Contour ...,39.99,29.99,29.99,1.333444,False


In [30]:
# rounded to 2 digits after the point: price, ratio
products_merged["price"] = products_merged["price"].round(2)
products_merged["ratio"] = products_merged["ratio"].round(2)

In [31]:
products_merged

,sku,name,desc,price,in_stock,type,id,order_id,product_quantity,unit_price,orderlines_date,median_price,price_ratio,false_price,ratio
0,OTT0133,Otterbox iPhone Case Symmetry 2.0 SE / 5s / 5 ...,resistant cover and thin beveled edges for iPh...,34.99,0.0,11865403.0,1119109,299539,1,18.99,2017-01-01 00:07:19,19.990,0.949975,False,1.75
1,LGE0043,"27UD58-B LG Monitor 27 ""4K UHD DisplayPort",Monitor for gamers and multimedia professional...,429.00,0.0,1296.0,1119110,299540,1,399.00,2017-01-01 00:19:45,375.000,1.064000,False,1.14
2,PAR0071,Parrot Bebop 2 White + Command FLYPAD and FPV ...,cuadricóptero wireless remote control with 25 ...,699.00,0.0,11905404.0,1119111,299541,1,474.05,2017-01-01 00:20:57,561.980,0.843535,False,1.24
3,WDT0315,"Blue WD 2TB Hard Drive 35 ""Mac and PC",Internal Hard Drive Western Digital 2TB 3.5-in...,79.00,0.0,12655397.0,1119112,299542,1,68.39,2017-01-01 00:51:40,70.785,0.966165,False,1.12
4,JBL0104,Gray Bluetooth Speaker JBL GO,Compact Bluetooth Handsfree for iPhone iPad an...,29.90,1.0,5398.0,1119113,299543,1,23.74,2017-01-01 01:06:38,24.490,0.969375,True,1.22
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
286192,JBL0122,JBL T450 BT Bluetooth Headset Black,Wireless headphones with folding design with 1...,49.95,1.0,5384.0,1650198,527397,1,42.99,2018-03-14 13:56:38,42.990,1.000000,False,1.16
286193,JBL0122,JBL T450 BT Bluetooth Headset Black,Wireless headphones with folding design with 1...,49.95,1.0,5384.0,1650199,527398,1,42.99,2018-03-14 13:57:25,42.990,1.000000,False,1.16
286194,PAC0653,Samsung SSD 850 expansion kit EVO 250GB + Data...,SSD upgrade kit 2008-2010 250 GB MacBook and M...,215.98,1.0,1433.0,1650200,527399,1,141.58,2018-03-14 13:57:34,151.660,0.933536,False,1.42
286195,APP0698,Apple Lightning Cable Connector to USB 1m Whit...,Apple Lightning USB Cable 1 meter to charge an...,25.00,1.0,1230.0,1650201,527400,2,9.99,2018-03-14 13:57:41,9.990,1.000000,False,2.50


In [32]:
orders_df = orders_df.dropna()
orders_df["created_date"] = pd.to_datetime(orders_df["created_date"])

In [33]:
# create short column with the first 3 char of sku
products_merged["short"] = products_merged["sku"].str[:3]

In [34]:
full_df = products_merged.merge(brands_df, on = "short", how = "left")

In [35]:
full_df = orders_df.merge(full_df, on = "order_id", how = "right")

In [36]:
full_df.dropna(subset=["state"])

,order_id,created_date,total_paid,state,sku,name,desc,price,in_stock,type,id,product_quantity,unit_price,orderlines_date,median_price,price_ratio,false_price,ratio,short,long
0,299539,2017-01-01 00:07:19,18.99,Shopping Basket,OTT0133,Otterbox iPhone Case Symmetry 2.0 SE / 5s / 5 ...,resistant cover and thin beveled edges for iPh...,34.99,0.0,11865403.0,1119109,1,18.99,2017-01-01 00:07:19,19.990,0.949975,False,1.75,OTT,Otterbox
1,299540,2017-01-01 00:19:45,399.00,Shopping Basket,LGE0043,"27UD58-B LG Monitor 27 ""4K UHD DisplayPort",Monitor for gamers and multimedia professional...,429.00,0.0,1296.0,1119110,1,399.00,2017-01-01 00:19:45,375.000,1.064000,False,1.14,LGE,LG
2,299541,2017-01-01 00:20:57,474.05,Shopping Basket,PAR0071,Parrot Bebop 2 White + Command FLYPAD and FPV ...,cuadricóptero wireless remote control with 25 ...,699.00,0.0,11905404.0,1119111,1,474.05,2017-01-01 00:20:57,561.980,0.843535,False,1.24,PAR,Parrot
3,299542,2017-01-01 00:51:40,68.39,Shopping Basket,WDT0315,"Blue WD 2TB Hard Drive 35 ""Mac and PC",Internal Hard Drive Western Digital 2TB 3.5-in...,79.00,0.0,12655397.0,1119112,1,68.39,2017-01-01 00:51:40,70.785,0.966165,False,1.12,WDT,Western Digital
4,299543,2017-01-01 01:06:38,23.74,Shopping Basket,JBL0104,Gray Bluetooth Speaker JBL GO,Compact Bluetooth Handsfree for iPhone iPad an...,29.90,1.0,5398.0,1119113,1,23.74,2017-01-01 01:06:38,24.490,0.969375,True,1.22,JBL,JBL
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
284032,527397,2018-03-14 13:56:38,42.99,Place Order,JBL0122,JBL T450 BT Bluetooth Headset Black,Wireless headphones with folding design with 1...,49.95,1.0,5384.0,1650198,1,42.99,2018-03-14 13:56:38,42.990,1.000000,False,1.16,JBL,JBL
284033,527398,2018-03-14 13:57:25,42.99,Shopping Basket,JBL0122,JBL T450 BT Bluetooth Headset Black,Wireless headphones with folding design with 1...,49.95,1.0,5384.0,1650199,1,42.99,2018-03-14 13:57:25,42.990,1.000000,False,1.16,JBL,JBL
284034,527399,2018-03-14 13:57:34,141.58,Shopping Basket,PAC0653,Samsung SSD 850 expansion kit EVO 250GB + Data...,SSD upgrade kit 2008-2010 250 GB MacBook and M...,215.98,1.0,1433.0,1650200,1,141.58,2018-03-14 13:57:34,151.660,0.933536,False,1.42,PAC,Pack
284035,527400,2018-03-14 13:57:41,19.98,Shopping Basket,APP0698,Apple Lightning Cable Connector to USB 1m Whit...,Apple Lightning USB Cable 1 meter to charge an...,25.00,1.0,1230.0,1650201,2,9.99,2018-03-14 13:57:41,9.990,1.000000,False,2.50,APP,Apple


In [37]:
full_df.info()

<class 'pandas.DataFrame'>
RangeIndex: 284037 entries, 0 to 284036
Data columns (total 20 columns):
 #   Column            Non-Null Count   Dtype         
---  ------            --------------   -----         
 0   order_id          284037 non-null  int64         
 1   created_date      283814 non-null  datetime64[us]
 2   total_paid        283814 non-null  float64       
 3   state             283814 non-null  str           
 4   sku               284037 non-null  str           
 5   name              284037 non-null  str           
 6   desc              284037 non-null  str           
 7   price             284037 non-null  float64       
 8   in_stock          284037 non-null  float64       
 9   type              283858 non-null  float64       
 10  id                284037 non-null  int64         
 11  product_quantity  284037 non-null  int64         
 12  unit_price        284037 non-null  float64       
 13  orderlines_date   284037 non-null  datetime64[us]
 14  median_price   

In [38]:
full_df.loc[full_df["short"] == "KEN", "long"] = "Kensington"
full_df.loc[full_df["short"] == "MUJ", "long"] = "Mujo"

In [39]:
full_df[["short", "long"]].loc[full_df["short"] == "MUJ"]

,short,long
1090,MUJ,Mujo
6835,MUJ,Mujo
14451,MUJ,Mujo
14464,MUJ,Mujo
15079,MUJ,Mujo
...,...,...
270496,MUJ,Mujo
270715,MUJ,Mujo
270900,MUJ,Mujo
275846,MUJ,Mujo


In [40]:
full_df.to_csv("data/processed/full_df.csv", index = False)